In [17]:
!pip install zss -q
!pip install antlr4-python3-runtime==4.11 -q

In [18]:
from kaggle_secrets import UserSecretsClient
secret_label = "GITHUB_TOKEN"
secret_value = UserSecretsClient().get_secret(secret_label)

In [19]:
import os

token = secret_value

repo_url = f"https://{token}@github.com/smiling-demon/llm-symbolic-ode-reasoning-dev.git"
repo_dir = "llm-symbolic-ode-reasoning-dev"

if not os.path.exists(repo_dir):
    !git clone {repo_url}

os.chdir(repo_dir)

!pwd

Cloning into 'llm-symbolic-ode-reasoning-dev'...
remote: Enumerating objects: 135, done.
remote: Counting objects: 100% (135/135), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 135 (delta 54), reused 112 (delta 31), pack-reused 0 (from 0)
Receiving objects: 100% (135/135), 1.53 MiB | 24.07 MiB/s, done.
Resolving deltas: 100% (54/54), done.
/kaggle/working/llm-symbolic-ode-reasoning-dev/llm-symbolic-ode-reasoning-dev


In [20]:
import pandas as pd

from llm import LLM
from methods import baseline
from metrics import evaluate_predictions

In [21]:
df = pd.read_excel("/kaggle/input/datasets/dmitrylessy/ode-basic-dataset/data/test.xlsx")

equations = df['equation'].tolist()
solutions = df['solution'].tolist()

In [22]:
llm = LLM("Qwen/Qwen2.5-3B-Instruct")

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [23]:
%%time

batch_size = 10
predictions = []

for i in range(0, len(equations), batch_size):
    predictions += baseline(llm, equations[i: i + batch_size], max_new_tokens=512)

CPU times: user 1min 3s, sys: 48.4 ms, total: 1min 4s
Wall time: 1min 3s


In [24]:
evaluation = evaluate_predictions(predictions, solutions)
for key in evaluation:
    evaluation[key] = [evaluation[key]]
pd.DataFrame.from_dict(evaluation)

,bleu,ast_error_size,ast_tree_similarity,exact_match
0,0.536454,0.79962,0.050919,0.06
